In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import os
from langchain_typesafe import Choice, Noul, Score, TypeSafeClassifier

In [27]:
classifier = TypeSafeClassifier(api_key=os.getenv("TYPE_SAFE_API_KEY"))

In [4]:
res = classifier.invoke(
    {
        "state": (
            "The deploy failed twice and customers are seeing 500s. "
            "Can someone look now?"
        ),
        "questions": {
            "is_urgent" : Noul(instructions="Does the customer is happy?"),
        }
    }
)

In [5]:
# NO - YES ? 0 - 1

In [6]:
res.answers["is_urgent"].noul

0.02

###### Compare with LLM's


In [7]:
from langchain_groq import ChatGroq
llm = ChatGroq(model=os.getenv("GROQ_AI_MODEL"))

In [8]:
prompt = """This Question Asked by the user  question: 'The deploy failed twice and customers are seeing 500s. Can someone look now?' is urgent or not?"""

In [9]:
res = llm.invoke(prompt)

In [10]:
res.content

'The request is **urgent**.  \n\n- **Impact:** Customers are encountering 500 errors, which means the service is currently unavailable or malfunctioning for end‑users.  \n- **Frequency:** The deployment has already failed twice, indicating a repeated problem that needs immediate attention.  \n- **Time sensitivity:** The phrasing “Can someone look now?” explicitly asks for immediate investigation.\n\nBecause it involves a production‑affecting incident that is directly impacting customers, it should be treated as a high‑priority, urgent issue.'

In [15]:
# query = "I've contacted you three times and I'm still waiting."

# res = classifier.invoke({
#     "state":query,
#     "questions": {
#         "is_angry":Noul(instructions="The customer sounds frustated")
#     }
# })

query = "I want to cancel my movie ticket and want my money back."

res = classifier.invoke({
    "state":query,
    "questions": {
        "is_refund":Noul(instructions="The customer wants the refund")
    }
})

In [16]:
res.answers["is_refund"]

NoulAnswer(type='noul', noul=0.98)

###### Choice Option


In [49]:
quest  = " I paid Rs 21000 for Gen AI Coruse, amount is deucted from account. But the course access is still locked and live class starts in 20 mins."

In [41]:
res = classifier.invoke({
        "state":quest,
    "questions":{
        "team":Choice(
            instructions = "Which team should handle this message",
            criteria = {
                "billing":"Payments and Refunds",
                "technical": "course access, login video problem",
                "sales":"Pricing, corporate plans and group discounts",
                "general":"Anything else"
            },
        )
    },
})

In [42]:
res.answers["team"]

ChoiceAnswer(type='choice', choice='technical', probabilities={'general': 0.0, 'technical': 0.99, 'billing': 0.01, 'sales': 0.0}, confidence=0.98)

In [43]:
res.answers["team"].choice

'technical'

###### Score


In [50]:
res = classifier.invoke({
    "state" : question,
    "questions":{
        "urgency":Score(
            instructions="How urgent is this message?",
            criteria = ["Cant wait", "Handle Today", "Handle within an hour"]
        )
    } 
})

In [51]:
res.answers['urgency'].model_dump()

{'type': 'score',
 'score': 0.15,
 'legend': {0: 'Cant wait', 1: 'Handle Today', 2: 'Handle within an hour'},
 'probabilities': {0: 0.92, 1: 0.02, 2: 0.06},
 'confidence': 0.77}